In [ ]:
import numpy as np
# ------------------------------------------------------------
# Manual SVD using eigen-decomposition
# A A^T = U Sigma^2 U^T
# A^T A = V Sigma^2 V^T
# ------------------------------------------------------------

In [ ]:
def manual_svd(A, k=None, tolerance=1e-3):

    A = np.asarray(A, dtype=float)
    m, n = A.shape

    if k is None: k = min(m, n)

    # Use smaller covariance matrix
    if m <= n:
        C = A @ A.T
        left = True
    else:
        C = A.T @ A
        left = False

    # Eigen-decomposition
    eigenvalues, vectors = np.linalg.eigh(C)

    # Sort largest eigenvalues first
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    vectors = vectors[:, idx]

    # Singular values = sqrt(eigenvalues)
    S = np.sqrt(np.clip(eigenvalues, 0, None))

    # Remove near-zero singular values and keep k components
    valid = S > tolerance
    S = S[valid][:k]
    vectors = vectors[:, valid][:, :k]

    # Recover U and V
    if left:
        U = vectors
        V = (A.T @ U) / S
    else:
        V = vectors
        U = (A @ V) / S

    return U, S, V.T

In [ ]:
# Example user-item rating matrix
A = np.array([
    [5, 4, 0, 1],
    [4, 5, 1, 0],
    [1, 0, 5, 4],
    [0, 1, 4, 5],
    [5, 4, 1, 1]
], dtype=float)
print("Original Matrix:\n", A)

# Full SVD
U, S, Vt = manual_svd(A)

print("\nU:\n", np.round(U, 3))
print("\nSingular Values:\n", np.round(S, 3))
print("\nVt:\n", np.round(Vt, 3))

# Full reconstruction
A_reconstructed = U @ np.diag(S) @ Vt
print("\nFull Reconstruction:\n", np.round(A_reconstructed, 3))
full_error = np.sqrt(np.mean((A - A_reconstructed) ** 2))
print("\nFull reconstruction RMSE:", round(full_error, 8))

# Rank-2 truncated SVD
U2, S2, Vt2 = manual_svd(A, k=2)
A_rank2 = U2 @ np.diag(S2) @ Vt2
print("\nRank-2 Approximation:\n", np.round(A_rank2, 3))
rank2_error = np.sqrt(np.mean((A - A_rank2) ** 2))
print("\nRank-2 reconstruction RMSE:", round(rank2_error, 6))

Original Matrix:
 [[5. 4. 0. 1.]
 [4. 5. 1. 0.]
 [1. 0. 5. 4.]
 [0. 1. 4. 5.]
 [5. 4. 1. 1.]]

U:
 [[-0.514 -0.274 -0.307  0.5  ]
 [-0.51  -0.271  0.619 -0.5  ]
 [-0.305  0.636 -0.446 -0.5  ]
 [-0.301  0.639  0.48   0.5  ]
 [-0.541 -0.198 -0.307 -0.   ]]

Singular Values:
 [11.686  8.373  1.527  1.414]

Vt:
 [[-0.652 -0.605 -0.323 -0.323]
 [-0.335 -0.311  0.629  0.629]
 [-0.68   0.733 -0.    -0.   ]
 [-0.     0.    -0.707  0.707]]

Full Reconstruction:
 [[ 5.  4. -0.  1.]
 [ 4.  5.  1. -0.]
 [ 1. -0.  5.  4.]
 [-0.  1.  4.  5.]
 [ 5.  4.  1.  1.]]

Full reconstruction RMSE: 0.0

Rank-2 Approximation:
 [[4.681 4.343 0.5   0.5  ]
 [4.643 4.307 0.5   0.5  ]
 [0.537 0.499 4.5   4.5  ]
 [0.498 0.463 4.5   4.5  ]
 [4.681 4.343 1.    1.   ]]

Rank-2 reconstruction RMSE: 0.465375
